# 3.2 · 缺失值处理 / Missing Values

> **课程定位 / Where this fits**
> **Part 3 第 2 课**。3.1 的 EDA 发现 age 缺 20% / deck 缺 77%——现在解决它。**核心不是"用什么填"，而是先搞清"为什么缺"**（缺失机制决定正确做法）。
> Part 3, lesson 2. The key isn't *what* to impute — it's understanding *why* it's missing first.

> 💡 **面试相关 / Interview-relevant**
> - "MCAR / MAR / MNAR 区别" ★★★★★（缺失值必考第一题）
> - "均值填补的问题" ★★★★（低估方差 + 扭曲相关）
> - "怎么避免填补时的数据泄漏" ★★★★★（fit 只在 train）
> - "KNN / 迭代填补什么时候用" ★★★

---

## 学习目标 / Learning Objectives
1. 区分 **MCAR / MAR / MNAR** 三种机制，并知道各自能用什么方法。
2. 量化**均值填补的两宗罪**（缩小方差、扭曲相关）。
3. 用 **简单 / 分组 / KNN / 迭代(MICE)** 四档填补，并对比。
4. **缺失指示符**技巧（让模型知道"这里曾经缺失"）。
5. 全程**防泄漏**：填补器只 fit 训练集。

## 目录 / TOC
1. [三种缺失机制 ⭐](#1)
2. [🛳 数据 + 缺失地图](#2)
3. [删除：什么时候可以](#3)
4. [⚠ 均值填补的两宗罪](#4)
5. [分组填补：更聪明的简单法](#5)
6. [KNN 填补](#6)
7. [迭代填补 MICE ⭐](#7)
8. [缺失指示符](#8)
9. [⚠ 防泄漏：填补器只 fit train](#9)
10. [实战对比 + 决策树](#10)
11. [小结](#11)


<a id="1"></a>
## 1. 三种缺失机制 ⭐ / Three Missingness Mechanisms

**这是缺失值领域最重要的概念**——机制决定哪些方法有效、会不会引入偏差。

| 机制 | 全称 | 含义 | 例子 | 后果 |
|---|---|---|---|---|
| **MCAR** | Missing Completely At Random | 缺失**纯随机**，和任何变量无关 | 试管随机打碎 | 删除/填补都无偏，最幸运 |
| **MAR** | Missing At Random | 缺失依赖**其他观测到的**变量 | 老人不爱填收入（缺失依赖年龄）| 用相关变量填补可纠偏 ⭐ |
| **MNAR** | Missing Not At Random | 缺失依赖**自身未观测的**值 | 高收入者隐瞒收入（缺失依赖收入本身）| 最难，简单填补必偏 |

**判别口诀**：
- 缺失和谁都无关 → MCAR
- 缺失能被**别的列**预测 → MAR
- 缺失只能被**这列自己（看不到的真值）**预测 → MNAR

⚠ **关键**：**MCAR vs MAR 可以用数据检验，但 MNAR 无法从数据本身证实**——因为缺的值看不到。判断 MNAR 要靠领域知识。
You can test MCAR vs MAR from data, but MNAR can never be confirmed from the data alone — the missing values are unseen. It takes domain knowledge.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 构造三种机制, 真值已知, 看填补能不能恢复
# Construct all three mechanisms with known truth
n = 2000
age = rng.uniform(20, 80, n)
income = 20 + 0.8*age + rng.normal(0, 8, n)   # 收入与年龄正相关 / income correlates with age

def make_missing(mechanism):
    inc = income.copy()
    if mechanism == "MCAR":
        mask = rng.random(n) < 0.3                          # 纯随机
    elif mechanism == "MAR":
        mask = rng.random(n) < (age - 20)/60 * 0.6          # 老人更易缺 (依赖 age)
    else:  # MNAR
        mask = rng.random(n) < (income - income.min())/(income.max()-income.min()) * 0.6  # 高收入更易缺
    inc[mask] = np.nan
    return inc

data = {m: make_missing(m) for m in ["MCAR","MAR","MNAR"]}
print(f"真实 income 均值 = {income.mean():.2f}")
for m, inc in data.items():
    print(f"{m}: 缺失 {np.isnan(inc).mean():.0%}, 观测均值 = {np.nanmean(inc):.2f}")


**看观测均值的偏差**：
- MCAR / MAR：观测均值 ≈ 真均值（缺失不挑值）
- **MNAR：观测均值明显偏低**——高收入者系统性消失，剩下的天然偏低

这就是机制的实战意义：**MNAR 下任何只看观测数据的填补都会低估**。


<a id="2"></a>
## 2. 🛳 数据 + 缺失地图 / Missingness Map

回到 Titanic。**缺失模式可视化**比数字更能看出机制线索。


In [ ]:
df = sns.load_dataset("titanic")
print("缺失率:")
print((df.isna().mean()*100).round(1).sort_values(ascending=False).head())

# 缺失地图: 白条 = 缺失 / missingness map
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.heatmap(df[["age","deck","embarked","embark_town"]].isna(), cbar=False,
            yticklabels=False, cmap="viridis", ax=axes[0])
axes[0].set_title("missingness map (黄=缺失)")

# age 缺失是否依赖 pclass? (检验 MAR 线索) / does age-missing depend on pclass?
miss_by_class = df.assign(age_missing=df.age.isna()).groupby("pclass")["age_missing"].mean()
miss_by_class.plot(kind="bar", ax=axes[1])
axes[1].set_title("age 缺失率 by pclass — 三等舱更高 → MAR 线索")
axes[1].set_ylabel("missing rate")
plt.tight_layout(); plt.show()
print("\nage 缺失率随舱等变化 → 不是 MCAR, 至少是 MAR (可用 pclass 辅助填补)")


<a id="3"></a>
## 3. 删除：什么时候可以 / Deletion: When It's OK

| 方法 | 做法 | 何时用 |
|---|---|---|
| **listwise (整行删)** | 任一列缺失就删行 | MCAR + 缺失比例小（<5%）|
| **删列** | 整列丢弃 | 缺失率极高（>60-70%, 如 deck）|
| **pairwise** | 每个分析用各自可用的行 | 相关矩阵等，少用 |

⚠ **删除的代价**：
- 非 MCAR 下删行 = **引入偏差**（删掉的不是随机子集）
- 即使 MCAR，删行也**损失样本量**（多列各缺一点，listwise 可能删掉一大半）


In [ ]:
print(f"Titanic 原始: {len(df)} 行")
print(f"listwise 删除所有缺失: {len(df.dropna())} 行 (只剩 {len(df.dropna())/len(df):.0%}!) — 太狠")
print(f"只删 age 缺失: {len(df.dropna(subset=['age']))} 行")
print(f"删 deck 列(77%缺) + 删 age 缺失行: 合理方案")


<a id="4"></a>
## 4. ⚠ 均值填补的两宗罪 / The Two Sins of Mean Imputation

均值填补是最常见的，也是**最常被滥用的**。两个量化的危害：


In [ ]:
from sklearn.impute import SimpleImputer

# 用 MCAR 数据演示 (机制最干净, 排除机制偏差, 只看方法本身的问题)
inc_mcar = data["MCAR"].reshape(-1, 1)
observed = inc_mcar[~np.isnan(inc_mcar)]
mean_filled = SimpleImputer(strategy="mean").fit_transform(inc_mcar).ravel()

print("=== 罪一: 缩小方差 ===")
print(f"真实 std         = {income.std():.2f}")
print(f"观测(非缺) std   = {observed.std():.2f}")
print(f"均值填补后 std   = {mean_filled.std():.2f}  ← 被人为压低!")
print("原因: 填进去的全是同一个值(均值), 方差当然变小 → 低估不确定性")

# 罪二: 扭曲相关 (用 age-income 这对) / distorting correlation
df_corr = pd.DataFrame({"age": age, "income": data["MCAR"]})
true_corr = np.corrcoef(age, income)[0,1]
filled_corr = np.corrcoef(age, SimpleImputer(strategy="mean").fit_transform(df_corr[["income"]]).ravel())[0,1]
print(f"\n=== 罪二: 扭曲相关 ===")
print(f"真实 age-income 相关 = {true_corr:.3f}")
print(f"均值填补后相关        = {filled_corr:.3f}  ← 被稀释 (填补值与 age 无关)")


**两宗罪**：
1. **缩小方差**：填进去的都是同一个均值 → 人为降低离散度 → 后续置信区间/检验**过度自信**（2.5 的覆盖率会偏）
2. **扭曲相关**：填补值和其他变量无关 → **稀释真实相关** → 削弱模型能学到的关系

> 💡 均值填补不是不能用——**MCAR + 低缺失率 + 树模型**时影响小。但默认就用它是懒惰。
> Mean imputation is fine under MCAR + low missingness + tree models, but it's lazy as a default.


<a id="5"></a>
## 5. 分组填补：更聪明的简单法 / Group-wise Imputation

**利用 MAR**：缺失依赖 pclass → 就用**同舱等的中位数**填，而非全局中位数。0.3 节用过，这里讲清原理。


In [ ]:
df_g = df.copy()
df_g["age_global"] = df_g["age"].fillna(df_g["age"].median())
df_g["age_grouped"] = df_g.groupby("pclass")["age"].transform(lambda s: s.fillna(s.median()))

print("各舱等的年龄中位数 (差异明显 → 分组填补更准):")
print(df.groupby("pclass")["age"].median())
print(f"\n全局填补值: {df.age.median():.0f} (所有缺失都填这个)")
print("分组填补: 一等舱缺失填~37, 三等舱填~24 — 尊重了 MAR 结构")


<a id="6"></a>
## 6. KNN 填补 / KNN Imputation

**思路**：缺失样本的值 = 它最近 $k$ 个邻居（按其他特征算距离）的均值。比分组更细——用**所有相关特征**找邻居。


In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# 用数值列做 KNN 填补 / KNN imputation on numeric columns
num_cols = ["age","fare","pclass","sibsp","parch"]
X = df[num_cols].copy()

# KNN 对尺度敏感 → 先标准化 (3.4 节细讲) / KNN is scale-sensitive, standardize first
Xs = StandardScaler().fit_transform(X)
X_knn = KNNImputer(n_neighbors=5).fit_transform(Xs)

# 还原 age 列对比 / compare age column
age_knn = X_knn[:, 0] * X["age"].std() + X["age"].mean()  # 粗略反标准化示意
filled_idx = df["age"].isna()
print(f"age 缺失数: {filled_idx.sum()}")
print(f"KNN 填补的 age 范围: [{age_knn[filled_idx].min():.0f}, {age_knn[filled_idx].max():.0f}]")
print("KNN 优点: 填补值有变化(非常量) → 不缩方差; 用多特征找邻居 → 更准")
print("KNN 缺点: O(n²) 慢, 对尺度敏感(必须先标准化), 高维失效(维度诅咒)")


<a id="7"></a>
## 7. 迭代填补 MICE ⭐ / Iterative Imputation (MICE)

**最强的通用填补**：把"填补"变成"**用其他列预测缺失列**"的回归问题，**循环迭代**直到收敛。

```
MICE 算法:
  1. 先用均值粗填所有缺失 (初始化)
  2. 对每个有缺失的列 c:
       用其他所有列做特征, c 的非缺失值做标签, 训练回归
       预测 c 的缺失位置, 替换
  3. 重复 2 多轮, 直到填补值稳定
```

sklearn 的 `IterativeImputer`（受 R 的 MICE 启发）。


In [ ]:
from sklearn.experimental import enable_iterative_imputer  # 必须先 import 这个 / required
from sklearn.impute import IterativeImputer

X = df[["age","fare","pclass","sibsp","parch"]].copy()
mice = IterativeImputer(max_iter=10, random_state=0)
X_mice = mice.fit_transform(X)

age_mice = X_mice[df["age"].isna().values, 0]
print(f"MICE 填补的 age: 均值 {age_mice.mean():.1f}, std {age_mice.std():.1f}")
print(f"对比均值填补 std=0 (常量), MICE 保留了变异性")
print("\nMICE 优点: 利用全部列间关系, 填补值有合理变异, MAR 下接近无偏")
print("MICE 缺点: 慢, 假设线性关系(默认), 可能不收敛")


<a id="8"></a>
## 8. 缺失指示符 / Missing Indicator

**绝招**：填补的同时，**额外加一列 0/1 标记"这里原来缺失"**。让模型自己学"缺失本身是否有信息"。

**为什么有用**：如果缺失是 MNAR（缺失本身携带信息），指示符让模型抓住它。例：信用数据里"未填收入"可能就是高风险信号。
If missingness is informative (MNAR), the indicator lets the model use it. e.g. "income not provided" may itself signal risk.


In [ ]:
from sklearn.impute import SimpleImputer

# add_indicator=True: 填补 + 自动加缺失标记列 / impute + add missingness flags
imp = SimpleImputer(strategy="median", add_indicator=True)
X_ind = imp.fit_transform(df[["age","fare"]])
print(f"原 2 列 → 填补后 {X_ind.shape[1]} 列 (2 填补 + {X_ind.shape[1]-2} 个指示符)")
print(f"指示符列就是 age/fare 是否缺失的 0/1, 模型可以用它")

# 验证 "age 缺失" 是否和生还有关 (是否值得加指示符) / is missingness informative?
df_chk = df.assign(age_missing=df.age.isna())
print(f"\nage 缺失者生还率: {df_chk[df_chk.age_missing]['survived'].mean():.2%}")
print(f"age 已知者生还率: {df_chk[~df_chk.age_missing]['survived'].mean():.2%}")
print("→ 有差异! age 缺失本身携带信息 → 加指示符有价值")


<a id="9"></a>
## 9. ⚠ 防泄漏：填补器只 fit train / Prevent Leakage

**最致命也最常见的错误**：在**全部数据**上算填补值（如全局均值），再划分 train/test。
**The deadliest mistake**: computing imputation values on the *whole* dataset before splitting.

为什么是泄漏：test 集的信息（它的均值贡献）渗进了训练，**测试性能虚高**。

✅ **正确姿势**：
1. 先划分 train/test
2. 填补器 **`.fit()` 只在 train**（学到 train 的均值/中位数/KNN 结构）
3. 用同一个填补器 **`.transform()`** train 和 test


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X = df[["age","fare"]]; y = df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

# ❌ 错误: 在全数据上填补 / WRONG: impute on all data
wrong = SimpleImputer(strategy="median").fit(X)  # fit 用了 X 全部 (含 test!)

# ✅ 正确: fit 只在 train / RIGHT: fit only on train
imp = SimpleImputer(strategy="median").fit(X_tr)   # 只看 train
X_tr_imp = imp.transform(X_tr)
X_te_imp = imp.transform(X_te)                      # test 用 train 学的中位数

print(f"全数据 age 中位数 (错误用):  {wrong.statistics_[0]:.1f}")
print(f"仅 train age 中位数 (正确):   {imp.statistics_[0]:.1f}")
print("差异虽小, 但原则必须遵守 — test 集的任何统计量都不能渗入训练")
print("\n💡 用 Pipeline (3.12) 自动保证这一点, 永不手滑")


<a id="10"></a>
## 10. 实战对比 + 决策树 / Comparison + Decision Tree

用已知真值的 MAR 数据，对比各方法**恢复真实分布**的能力：


In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler

# MAR 数据 (缺失依赖 age, 用 age 辅助可纠偏) / MAR data
df_mar = pd.DataFrame({"age": age, "income": data["MAR"]})
true_mean, true_std = income.mean(), income.std()

results = {}
# 均值
results["mean"] = SimpleImputer(strategy="mean").fit_transform(df_mar[["income"]]).ravel()
# KNN (用 age 找邻居)
Xs = StandardScaler().fit_transform(df_mar)
results["KNN(用age)"] = KNNImputer(n_neighbors=10).fit_transform(Xs)[:,1]*df_mar.income.std()+np.nanmean(df_mar.income)
# MICE (用 age 回归)
results["MICE(用age)"] = IterativeImputer(random_state=0).fit_transform(df_mar)[:,1]

print(f"真实: mean={true_mean:.2f}, std={true_std:.2f}\n")
print(f"{'方法':<14} {'mean':>8} {'std':>8}")
print(f"{'观测(非缺)':<16} {np.nanmean(df_mar.income):>8.2f} {np.nanstd(df_mar.income):>8.2f}")
for k, v in results.items():
    print(f"{k:<16} {v.mean():>8.2f} {v.std():>8.2f}")
print("\nMICE/KNN 利用 age 纠正了 MAR 偏差并保留了方差; 均值填补缩小了 std")


### 缺失值处理决策树

```
1. 先问机制 (EDA + 领域知识)
   MCAR? MAR? MNAR?
2. 缺失率 > 60-70%? → 删列 (或转成"是否缺失"二值)
3. 缺失率 < 5% + MCAR? → 删行 or 简单填补都行
4. 数值列, MAR:
     有明显分组变量 → 分组中位数 (简单有效)
     特征间关系强   → MICE / KNN
5. 类别列 → 众数 or 新增 "Missing" 类别
6. 怀疑 MNAR / 缺失可能有信息 → 加缺失指示符
7. 永远: 填补器只 fit train (用 Pipeline 保证)
```


<a id="11"></a>
## 11. 小结 / Summary

```
机制决定一切 ⭐:
  MCAR (纯随机) — 删/填都无偏
  MAR  (依赖其他观测列) — 用相关列填可纠偏 (KNN/MICE/分组)
  MNAR (依赖自身真值) — 简单填必偏, 加指示符 + 领域知识
均值填补两宗罪: 缩方差 + 扭曲相关
方法谱: 删除 < 均值 < 分组中位数 < KNN < MICE
缺失指示符: 让模型用"缺失本身的信息"
防泄漏: 填补器 .fit() 只在 train ⭐
```

### 💡 面试速查
1. **MCAR/MAR/MNAR** 定义 + "MNAR 无法从数据证实"
2. **均值填补缩方差 + 扭相关**
3. **MAR 下用相关列填补**（分组/KNN/MICE）可纠偏
4. **填补只 fit train** — 否则泄漏
5. **缺失指示符** 处理信息性缺失

### 下一节
**3.3 异常值检测**——缺失处理好了，下一个脏数据问题：异常值。z-score / IQR / Isolation Forest / LOF / 马氏距离。
